# 08 · Joint demand × service analysis

Este notebook responde **Q5**: a piora de serviço ajuda a explicar a queda de demanda em horas chuvosas?

**Janela usada aqui:** a janela integrada de **19 dias**.
**Regra de interpretação:** esta análise continua sendo **observacional / exploratória**, sem reivindicação causal forte.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
integrated = pd.read_parquet(DERIVED / 'integrated_route_hour.parquet')
model_df = integrated.loc[integrated['coverage_flag'].fillna('ok') != 'partial_day'].copy()
model_df = model_df.dropna(
    subset=['boardings', 'rain_mm', 'headway_p50', 'speed_p50', 'service_gap_index', 'hour', 'day_of_week', 'route_norm']
).copy()

print('=== Modeling sample ===')
print('rows:', len(model_df))
print('routes:', model_df['route_norm'].nunique())
print('days:', pd.to_datetime(model_df['date']).dt.date.nunique())
print(model_df[['date', 'hour', 'route_norm', 'boardings', 'rain_mm', 'headway_p50', 'speed_p50', 'service_gap_index']].head(10).to_string(index=False))


In [ ]:
base_model = smf.glm(
    'boardings ~ rain_mm + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

service_model = smf.glm(
    'boardings ~ rain_mm + headway_p50 + speed_p50 + service_gap_index + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

interaction_model = smf.glm(
    'boardings ~ rain_mm + headway_p50 + speed_p50 + service_gap_index + rain_mm:headway_p50 + rain_mm:speed_p50 + rain_mm:service_gap_index + C(route_norm) + C(hour) + C(day_of_week)',
    data=model_df,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

comparison = pd.DataFrame(
    {
        'model': ['base', 'service', 'interaction'],
        'rain_coef': [base_model.params['rain_mm'], service_model.params['rain_mm'], interaction_model.params['rain_mm']],
        'rain_pvalue': [base_model.pvalues['rain_mm'], service_model.pvalues['rain_mm'], interaction_model.pvalues['rain_mm']],
        'aic': [base_model.aic, service_model.aic, interaction_model.aic],
        'pct_effect_per_mm': [100 * (np.exp(base_model.params['rain_mm']) - 1), 100 * (np.exp(service_model.params['rain_mm']) - 1), 100 * (np.exp(interaction_model.params['rain_mm']) - 1)],
    }
)
print('=== Nested model comparison ===')
print(comparison.to_string(index=False))
print()

interaction_terms = pd.DataFrame(
    {
        'term': ['rain_mm:headway_p50', 'rain_mm:speed_p50', 'rain_mm:service_gap_index'],
        'coef': [interaction_model.params.get('rain_mm:headway_p50', np.nan), interaction_model.params.get('rain_mm:speed_p50', np.nan), interaction_model.params.get('rain_mm:service_gap_index', np.nan)],
        'pvalue': [interaction_model.pvalues.get('rain_mm:headway_p50', np.nan), interaction_model.pvalues.get('rain_mm:speed_p50', np.nan), interaction_model.pvalues.get('rain_mm:service_gap_index', np.nan)],
    }
)
print('=== Interaction terms ===')
print(interaction_terms.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
sns.barplot(data=comparison, x='model', y='pct_effect_per_mm', ax=axes[0])
axes[0].axhline(0, color='black', linewidth=1)
axes[0].set_title('Q5 · Rain effect across nested models')
axes[0].set_ylabel('Percent effect of 1 mm rain')

sns.barplot(data=interaction_terms, x='term', y='coef', ax=axes[1])
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_title('Q5 · Interaction coefficients')
axes[1].tick_params(axis='x', rotation=20)
savefig('q5_nested_model_comparison.png')
plt.show()


## Leitura preliminar

- Se `rain_mm` perde magnitude quando as variáveis operacionais entram no modelo, parte da queda de demanda é compatível com **mediação via piora operacional**.
- Se um termo de interação fica significativo, isso é o indício mais forte para a leitura de **amplificação**.
- Mesmo assim, o resultado precisa ser comunicado como **evidência associativa** dentro da janela integrada de 19 dias.
